# 03_NumPy与图像

**学习目标：** 读懂shape/dtype/axis，完成HWC到CTHW转换，理解像素缩放。

适合：读过CS50P但基础不熟、刚开始深度学习。按顺序学习；每课先看讲解、跑示例、做练习，再查答案。


## 本课要点

shape描述各轴长度，dtype描述每个数的类型，axis指定沿哪一个维度运算。普通图片常是[H,W,C]，本题视频输入是[C,T,H,W]。Depth_Color三通道是伪彩深度表示，IR一通道，共四通道；不能把伪彩像素直接当距离米数。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 图片是数字数组


In [ ]:
import numpy as np
from PIL import Image
from depth_ir_io import sample_frame_lists
path, _, _ = L.sample_path('train', 0)
depth, ir = sample_frame_lists(path)
with Image.open(depth[0]) as image:
    image_array = np.asarray(image.convert('RGB'))
print(image_array.shape, image_array.dtype, image_array.min(), image_array.max())
chw = image_array.transpose(2, 0, 1).astype('float32') / 255
print('CHW：', chw.shape, chw.dtype, chw.min(), chw.max())


## 沿时间堆叠并加入IR通道


In [ ]:
clip, label, classes = L.load_clip('train', 0)
print('CTHW：', clip.shape, '标签：', label)
assert clip.shape == (4, 16, 128, 128)
assert clip.dtype == np.float32 and 0 <= clip.min() <= clip.max() <= 1
print('每个通道均值：', clip.mean(axis=(1, 2, 3)))


## 先用小数组理解axis


In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print('每列平均(axis=0)：', a.mean(axis=0))
print('每行平均(axis=1)：', a.mean(axis=1))
print('transpose交换轴；reshape重新解释排列，二者不能随意互换。')


## 动手练习

对视频每一帧分别求空间平均，保留通道轴与时间轴，得到[4,16]数组。

显示“待完成”表示练习尚未作答；代码示例运行成功不等于练习完成。


In [ ]:
your_answer = None  # TODO：clip.mean(axis=...)
if your_answer is None:
    print('待完成：只消去H与W两个轴')
else:
    assert your_answer.shape == (4, 16)
    np.testing.assert_allclose(your_answer[0, 0], clip[0, 0].mean(), rtol=1e-6)
    print('练习通过')


## 常见误区

用uint8做减法可能溢出；先转float32。模型内还会标准化，/255只完成像素缩放。


## 过关检查与选读

口述[4,16,128,128]每个数字；解释axis=(2,3)；区分transpose与reshape。

选做：改变示例中一个参数，先预测现象再运行，并记下原因。

[对应参考答案](../参考答案/03_NumPy与图像.ipynb) · [官方选读](https://numpy.org/doc/stable/user/absolute_beginners.html)

完成后在README.md#学习记录写下一句自己的解释。
